## Phase 3 - Data Preparation

This is where we prepare the data for Phase 4 (modeling).

Exploratory data analysis already highlighted the variables most correlated with the outcome: complaints_count (0.45) and delivery_delay_days (0.40).

In this phase we will:
- Drop irrelevant columns and columns at risk of data leakage.
- Apply one-hot encoding to the categorical (text) variable.
- Separate features and target.
- Split the data into training and test sets.  


In [8]:
# Libraries

import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

# Load CSV file
# Create target variable

df = pd.read_csv("desafio_nps_fase_1.csv")

df["nps_detrator"] = (df['nps_score'] <= 6).astype(int)




### Dropping variables

We remove low-signal variables and those that are not allowed because of data leakage (`csat_internal_score`, `repeat_purchase_30d`).

`nps_score` is dropped because, as explained in Phase 1 (Business Understanding), we do not observe this field until the end of the customer journey—so we treat it as leakage.

Customer journey: order placed → payment → picking → shipping → delivery → NPS survey 


In [9]:
columns_to_drop = ['customer_id', 'order_id', 'csat_internal_score', 'repeat_purchase_30d', 'nps_score']

df = df.drop(columns_to_drop, axis=1)
df.head()


,customer_age,customer_region,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count,nps_detrator
0,63,Nordeste,14,139.73,4,39.35,4,2,2,55.53,3,0,4,3,0
1,20,Sul,1,458.95,2,9.51,10,6,4,28.23,3,0,10,3,1
2,46,Nordeste,111,507.06,5,42.82,6,6,1,40.99,1,4,5,7,1
3,52,Centro-Oeste,117,302.19,2,19.58,9,5,2,35.24,3,1,11,4,1
4,56,Norte,50,253.06,1,29.37,11,13,1,39.32,1,1,0,3,0


### Handling categorical variables

Machine learning models do not understand raw text like "Sudeste" or "Sul". We turn that column into numbers using **one-hot encoding**.

**What is one-hot encoding?**  
It turns one categorical column into several binary columns (0 or 1).

**Why use it?**  
We found **no meaningful regional bias**, so `customer_region` will probably not be a very important feature—but we still include it and let the model decide.

**What if there were strong regional bias?**  
The model would put more weight on some regions. For example, if North were 90% detractors and South 50%, the model would learn that region is a strong predictor and the company might act regionally.


With no regional bias, the model will likely learn that `customer_region` has low predictive power.



In [10]:
# One-hot encode region (k-1 columns with drop_first=True)
df = pd.get_dummies(df, columns=['customer_region'], drop_first=True)

df.columns


Index(['customer_age', 'customer_tenure_months', 'order_value',
       'items_quantity', 'discount_value', 'payment_installments',
       'delivery_time_days', 'delivery_delay_days', 'freight_value',
       'delivery_attempts', 'customer_service_contacts',
       'resolution_time_days', 'complaints_count', 'nps_detrator',
       'customer_region_Nordeste', 'customer_region_Norte',
       'customer_region_Sudeste', 'customer_region_Sul'],
      dtype='str')

In [11]:
# Preview first rows after encoding
df.head(10)


,customer_age,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count,nps_detrator,customer_region_Nordeste,customer_region_Norte,customer_region_Sudeste,customer_region_Sul
0,63,14,139.73,4,39.35,4,2,2,55.53,3,0,4,3,0,True,False,False,False
1,20,1,458.95,2,9.51,10,6,4,28.23,3,0,10,3,1,False,False,False,True
2,46,111,507.06,5,42.82,6,6,1,40.99,1,4,5,7,1,True,False,False,False
3,52,117,302.19,2,19.58,9,5,2,35.24,3,1,11,4,1,False,False,False,False
4,56,50,253.06,1,29.37,11,13,1,39.32,1,1,0,3,0,False,True,False,False
5,35,75,568.76,6,36.58,3,4,5,41.82,2,2,3,5,1,False,False,True,False
6,37,68,41.29,3,99.62,6,8,3,35.83,3,3,4,6,1,False,False,True,False
7,60,37,428.76,4,29.54,10,11,5,44.50,1,0,2,2,1,False,False,False,True
8,40,60,121.56,3,91.95,6,6,3,24.88,2,1,9,3,0,False,False,False,True
9,51,70,411.01,6,37.47,3,9,2,30.59,1,0,7,2,1,False,False,True,False


### Splitting features and target

X → all columns the model uses to learn (features)  
y → the column the model predicts (target)  

Features = all columns in `df` except `nps_detrator` — `nps_score` was already removed in the previous step because of leakage risk.

Target = `nps_detrator`  




In [12]:
# Feature matrix and target vector
X = df.drop(columns=['nps_detrator'])
y = df['nps_detrator']

print(X.shape)
print(y.shape)


(2500, 17)
(2500,)


### Splitting the dataset into training and test sets

We split the data into two groups:

Training → the model learns from this split  
Test → the model is evaluated on data it has never seen


In [13]:
from sklearn.model_selection import train_test_split

# 80% train / 20% test (reproducible split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)


(2000, 17)
(500, 17)


Training → 2,000 rows (80% of the data): the model learns from this  
Test → 500 rows (20% of the data): the model is evaluated on this

An 80/20 split is the most common choice for medium-sized datasets: enough data for training without sacrificing evaluation quality.  
`random_state=42` makes the split reproducible—the partition is the same on every run.


Preparing the data for Phase 4 — Data Modeling


In [14]:
# Save prepared splits for Phase 4 — Data Modeling

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)



With the data prepared, the next step is Phase 4: modeling.
